# Laboratorio 2.1 · Pipelines ETL en Python: de bronze a silver

**Curso:** Pipelines de Datos y ETL con Python · BSG Institute
**Caso integrador:** la tienda. Ya ingestamos (1.2 y 1.3) y ya sabemos procesar rápido (1.4).
Hoy hacemos que los datos **digan la verdad**.

Bronze tiene 14 cargas diarias (del 8 al 21 de septiembre de 2026) de cuatro fuentes:

| Tabla de bronze | De dónde viene | Cómo llega |
|---|---|---|
| `pedidos_web` | Base de la tienda en línea (extracción incremental, 1.3) | Con tipos, pero con versiones repetidas |
| `pedidos_tienda` | Punto de venta de 3 sucursales | **Todo es texto**, hora local, folios propios |
| `clientes` | CRM, foto completa diaria | Cada quien escribe la ciudad como quiere |
| `tipo_cambio` | API de tipo de cambio | Solo días hábiles, ventanas que se traslapan |

| Parte | Qué haces | La pregunta |
|---|---|---|
| 1 | Mirar bronze | ¿Cuántos pedidos hay? (no es la cantidad de filas) |
| 2 | Diseño modular | ¿Por qué cada transformación es una función pura? |
| 3 | Limpiar y conformar | ¿Cómo se vuelven una sola tabla dos sistemas distintos? |
| 4 | Fecha de negocio | ¿De qué día es una venta de las 9 de la noche? |
| 5 | Deduplicar | ¿Con cuál versión me quedo? |
| 6 | Enriquecer con tipo de cambio | ¿Por qué el join obvio pierde casi un tercio de las ventas? |
| 7 | Pruebas con pytest | ¿Cómo sé que no rompí nada? |

## Preparación

In [ ]:
%%capture
!pip -q install -U "polars>=1.10" "pandera[polars]>=0.20" pyarrow pytest

In [ ]:
# @title Instalar el paquete del kit (el mismo código que el kit local)
import base64, io, zipfile
zipfile.ZipFile(io.BytesIO(base64.b64decode("..."))).extractall(".")
print("Paquete transformacion instalado")

In [ ]:
import inspect, warnings
from datetime import date
import polars as pl
from IPython.display import Code, display
from transformacion import generar_bronze, leer, limpieza, dedup, enriquecer, scd, calidad, gold, pipeline

warnings.filterwarnings("ignore", category=UserWarning)
pl.Config(tbl_rows=16, tbl_cols=12, fmt_str_lengths=40, tbl_width_chars=160)

def ver(funcion):
    display(Code(inspect.getsource(funcion), language="python"))

LANDING = "landing"
print(generar_bronze.generar(LANDING))     # 14 cargas: del 08/09 al 21/09/2026
b = leer.todas(LANDING)                      # un DataFrame por tabla de bronze
print("Polars", pl.__version__)

## Parte 1 · Bronze, tal como llegó

Bronze es una copia fiel de lo que mandaron los sistemas de origen, particionada por `fecha_carga`.
Nadie la corrige a mano: si algo está mal, se arregla **en la transformación**, y queda registrado en el código.

In [ ]:
!find landing/bronze -maxdepth 2 | sort | head -8
pl.DataFrame({"tabla": list(b), "filas": [len(df) for df in b.values()]})

In [ ]:
web = b["pedidos_web"]
print(f"Filas en bronze: {len(web):,}   ·   pedidos distintos (id): {web['id'].n_unique():,}")
web.sort("id", "fecha_carga").filter(pl.col("id") == web["id"][25_000])

El mismo pedido aparece en varias cargas: la extracción incremental lo trae cada vez que **cambia** (pagado, enviado, entregado).
Bronze no es una tabla de pedidos: es una **bitácora de extracciones**.

Ahora, las mañas de cada fuente:

In [ ]:
display(web["estado"].value_counts().sort("count", descending=True))
display(web["moneda"].value_counts().sort("count", descending=True))
b["pedidos_tienda"].head(4)

- `estado`: el mismo valor escrito con mayúsculas, espacios y combinaciones.
- `moneda`: `MN` (moneda nacional) y `mxn` también son pesos.
- El punto de venta manda **texto**: `$1,234.50`, la fecha como `dd/mm/aaaa`, **sin zona horaria**, y el cliente vacío en las ventas de mostrador.

## Parte 2 · Diseño modular: una transformación es una función

La regla del kit: **cada transformación recibe un DataFrame y devuelve otro**. No lee archivos, no escribe archivos,
no pregunta la hora del sistema. Leer está en `leer.py`; escribir, en `pipeline.py`. Todo lo de en medio es lógica pura.

¿Por qué tanto cuidado? Porque una función pura se prueba con tres filas hechas a mano, se reutiliza en otro pipeline
y da el mismo resultado hoy y dentro de un año.

In [ ]:
ver(limpieza.texto)
ver(limpieza.estado)

In [ ]:
prueba = pl.DataFrame({"estado": ["PAGADO ", " enviado", "Entregado", "pagdo", None]})
prueba.with_columns(limpio=limpieza.estado())

`pagdo` queda **nulo**: lo que no se reconoce no se adivina. En la sesión 2.2 esa fila irá a cuarentena con su motivo.
Adivinar «seguro quiso decir pagado» es como se cuelan errores que nadie puede rastrear después.

## Parte 3 · Limpiar y conformar: dos sistemas, un solo esquema

**Conformar** es llevar cada fuente al esquema de silver: mismos nombres, mismos tipos, mismas reglas.
Así se ve el del punto de venta:

In [ ]:
ver(limpieza.conformar_tienda)

In [ ]:
conformados = limpieza.pedidos(b["pedidos_web"], b["pedidos_tienda"])
print(conformados.schema)
conformados.filter(pl.col("pedido_id") == 1).unique(["origen", "pedido_id"]).select("origen", "pedido_id", "estado", "total", "moneda", "canal", "creado_en")

El pedido **1** existe en las dos fuentes y son ventas distintas. Por eso la llave de negocio de silver es
**`(origen, pedido_id)`**, no `pedido_id` solo. Si unes las fuentes por `id`, mezclas pedidos de dos sistemas.

Antes y después de limpiar:

In [ ]:
antes = pl.concat([web.select(v=pl.col("estado")), b["pedidos_tienda"].select(v=pl.col("estatus"))])
print("Valores distintos de estado:", antes["v"].n_unique(), "→", conformados["estado"].n_unique())
print("Valores distintos de moneda:", pl.concat([web["moneda"], b["pedidos_tienda"]["moneda"]]).n_unique(), "→", conformados["moneda"].n_unique())
conformados["estado"].value_counts().sort("count", descending=True)

Con los clientes pasa lo mismo, pero con la ciudad:

In [ ]:
crudas = b["clientes"]
print("Formas distintas de escribir la ciudad:", crudas["ciudad"].n_unique())
(crudas.with_columns(canonica=limpieza.ciudad())
       .filter(pl.col("canonica") == "CDMX")["ciudad"].value_counts().sort("count", descending=True))

In [ ]:
print("Después de limpiar:", crudas.select(limpieza.ciudad())["ciudad"].n_unique(), "ciudades")

## Parte 4 · ¿De qué día es una venta de las 9 de la noche?

Las bases guardan las horas en **UTC**. La Ciudad de México está 6 horas atrás. Una venta de las 21:30 en CDMX
se guarda como las 03:30 **del día siguiente** en UTC. Si el reporte agrupa por la fecha UTC, esa venta cae en el día equivocado.

In [ ]:
ver(limpieza.fecha_negocio)

In [ ]:
fechas = conformados.select("origen", "pedido_id", "creado_en",
                            hora_cdmx=pl.col("creado_en").dt.convert_time_zone(limpieza.ZONA_NEGOCIO),
                            fecha_utc=pl.col("creado_en").dt.date(), fecha_negocio=limpieza.fecha_negocio())
distintas = fechas.filter(pl.col("fecha_utc") != pl.col("fecha_negocio"))
print(f"Pedidos cuya fecha UTC no es su fecha de negocio: {len(distintas):,} de {len(fechas):,} ({len(distintas) / len(fechas):.0%})")
distintas.head(4)

Una de cada cuatro ventas. Las del punto de venta llegaron como hora local **sin zona**: `conformar_tienda` les pone
la zona de CDMX y las convierte a UTC. Regla: **en silver todo se guarda en UTC; la fecha de negocio se calcula en la zona del negocio**.

## Parte 5 · Deduplicar: ¿con cuál versión me quedo?

Hay dos clases de repetidos:

- **Duplicados exactos:** la misma fila dos veces (un reintento que escribió dos veces, una ventana de traslape).
- **Versiones:** el mismo pedido con distinto estado en distintas cargas.

El reflejo de usar `.unique()` solo resuelve la primera.

In [ ]:
print(dedup.resumen(conformados))
ingenuo = conformados.unique()
print(f"Con .unique():          {len(ingenuo):,} filas")
unicos = dedup.ultima_version(conformados)
print(f"Con ultima_version():   {len(unicos):,} filas  (una por pedido)")

In [ ]:
ver(dedup.ultima_version)

El orden importa: primero la versión más reciente **en el origen** (`actualizado_en`). Si dos versiones tienen la misma
marca de tiempo, gana **la que se extrajo después** (`_extraido_en`).

¿Cuándo empatan? El 17/09 soporte corrigió 6 pedidos con un `UPDATE` manual que **no movió** `actualizado_en`, y los reextrajo.

In [ ]:
empates = (conformados.filter(pl.col("origen") == "web")
           .group_by("pedido_id", "actualizado_en").agg(pl.col("estado").n_unique().alias("estados"))
           .filter(pl.col("estados") > 1))
caso = empates["pedido_id"][0]
display(conformados.filter((pl.col("origen") == "web") & (pl.col("pedido_id") == caso))
        .select("pedido_id", "estado", "actualizado_en", "_extraido_en", "fecha_carga").sort("_extraido_en"))
print("Silver se queda con:", unicos.filter((pl.col("origen") == "web") & (pl.col("pedido_id") == caso))["estado"].item())

Sin el segundo criterio de orden, el resultado dependería de en qué orden leyó Polars los archivos: **a veces uno, a veces el otro**.
Un pipeline que no da siempre lo mismo con la misma entrada no es confiable, aunque «casi siempre» esté bien.

## Parte 6 · Enriquecer con el tipo de cambio

Una de cada diez ventas es en dólares. Para sumar ingresos hay que convertir con el tipo de cambio **del día de la venta**.
Primero, el tipo de cambio limpio: una tasa por fecha. El 14/09 el publicador **corrigió** el dato del 11/09:

In [ ]:
display(b["tipo_cambio"].filter(pl.col("fecha") == date(2026, 9, 11)).select("fecha", "tasa", "fecha_carga").unique().sort("fecha_carga"))
tc = enriquecer.tipo_cambio_diario(b["tipo_cambio"])
tc.filter(pl.col("fecha").is_between(date(2026, 9, 10), date(2026, 9, 18)))

Gana la carga más reciente (trae la corrección). Y fíjate qué fechas **no** están: 12 y 13 (fin de semana) y **16** (feriado).
Ahora el join obvio:

In [ ]:
validos = unicos.filter(pl.col("total").is_not_null())     # en la 2.2 esto lo hará la validación de calidad
ingenuo = enriquecer.con_tipo_cambio_ingenuo(validos, tc)
print(f"Join por fecha: {len(ingenuo):,} de {len(validos):,} pedidos  ->  se perdieron {len(validos) - len(ingenuo):,}")
(validos.with_columns(fecha_negocio=limpieza.fecha_negocio())
        .join(tc, left_on="fecha_negocio", right_on="fecha", how="anti")
        .with_columns(dia=pl.col("fecha_negocio").dt.weekday().replace_strict(
            {1: "lunes", 2: "martes", 3: "miércoles", 4: "jueves", 5: "viernes", 6: "sábado", 7: "domingo"}))
        ["dia"].value_counts().sort("count", descending=True))

Un `inner join` no avisa: **descarta**. El tablero mostraría que la tienda no vende en fin de semana ni el 16 de septiembre.
Y los pocos del lunes son ventas de la madrugada del 21: la tasa de ese día **todavía no se publica**.
La solución es un join **«as of»**: para cada venta, la última tasa publicada en o antes de su fecha.

In [ ]:
ver(enriquecer.con_tipo_cambio)

In [ ]:
enriquecidos = enriquecer.con_tipo_cambio(validos, tc)
print(f"Pedidos: {len(enriquecidos):,}   ·   sin tipo de cambio: {enriquecidos['total_mxn'].null_count()}")
(enriquecidos.filter(pl.col("moneda") == "USD")
             .select("pedido_id", "fecha_negocio", "fecha_tc", "dias_tc", "total", "mxn_por_usd", "total_mxn")
             .filter(pl.col("dias_tc") > 0).head(4))

¿Cuánto cambia el ingreso si nadie convierte?

In [ ]:
pagados = enriquecidos.filter(pl.col("estado") != "cancelado")
convertido = pl.when(pl.col("moneda") == "USD").then(pl.col("total") * pl.col("mxn_por_usd")).otherwise("total")
calculos = {"Sumar total sin convertir": pagados["total"].sum(),
            "Join ingenuo (sin fines de semana)": ingenuo.filter(pl.col("estado") != "cancelado").select(convertido.sum()).item(),
            "Correcto: as of + conversión": pagados["total_mxn"].sum()}
correcto = calculos["Correcto: as of + conversión"]
pl.DataFrame({"cálculo": list(calculos), "ingreso_mxn": [f"{v:,.0f}" for v in calculos.values()],
              "diferencia": [f"{(v - correcto) / correcto:+.1%}" for v in calculos.values()]})

Ninguno de los dos errores truena: el pipeline termina «bien» y el número es falso. Por eso, en la 2.2,
vamos a medir lo que entra y lo que sale de cada paso.

## Parte 7 · Pruebas con pytest

Las funciones del kit tienen pruebas. Cada una arma un DataFrame chico a mano con el caso que importa:

In [ ]:
!sed -n '/def test_sabado/,/^$/p' tests/test_transformacion.py
!python -m pytest -q tests/test_transformacion.py -k "not pipeline" 2>&1 | tail -3

### Ejercicio · El punto de venta cambió su formato

A partir de mañana, el punto de venta mandará el importe como `MXN 1,234.50` en lugar de `$1,234.50`.
**Primero escribe la prueba** de lo que debería pasar:

In [ ]:
%%writefile tests/test_mio.py
import polars as pl
from transformacion import limpieza

def test_importe_con_prefijo_de_moneda():
    df = pl.DataFrame({"i": ["MXN 1,234.50", "$99.00", "USD 10.00"]})
    assert df.select(limpieza.importe("i"))["i"].to_list() == [1234.5, 99.0, 10.0]

In [ ]:
!python -m pytest -q tests/test_mio.py 2>&1 | tail -4

Falla: la función actual solo quita `$`, comas y espacios. Abre `transformacion/limpieza.py` en el panel de archivos
de Colab y corrige la función `importe` para que se quede solo con dígitos, punto y signo. Después vuelve a correr la celda de arriba.

Si te atoras, la celda siguiente aplica la solución.

In [ ]:
# @title Solución del ejercicio
from pathlib import Path
ruta = Path("transformacion/limpieza.py")
codigo = ruta.read_text(encoding="utf-8")
ruta.write_text(codigo.replace('str.replace_all(r"[$,\\s]", "")', 'str.replace_all(r"[^0-9.\\-]", "")'), encoding="utf-8")
!python -m pytest -q tests/test_mio.py tests/test_transformacion.py -k "not pipeline" 2>&1 | tail -2

Las dos suites pasan: arreglaste el caso nuevo **sin romper** los anteriores. Eso es lo que compran las pruebas.

## Cierre

| Hoy viste | Principio |
|---|---|
| Bronze es una bitácora | Se transforma, no se corrige a mano |
| Funciones puras | Leer y escribir en las orillas; la lógica en medio |
| Conformar | Una llave de negocio que incluye el origen |
| Fecha de negocio | Guardar en UTC, agrupar en la zona del negocio |
| Deduplicar | Última versión con un desempate determinista |
| Join «as of» | Un `inner join` descarta en silencio |
| pytest | La prueba primero, luego el arreglo |

**Próxima sesión (2.2):** historia de los clientes (SCD tipo 1 y 2), gold y **calidad de datos** con Pandera:
qué pasa cuando el origen manda basura.